<a href="https://colab.research.google.com/github/vifirsanova/ML-2026-pt-2/blob/main/tutorials/2_scraping/requests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Библиотека `requests` в Python: парсинг веб-страниц

## Что такое `requests`?

`requests` — это популярная библиотека Python для выполнения HTTP-запросов. С её помощью программа может общаться с веб-серверами: отправлять запросы, получать ответы, скачивать страницы и работать с API.

## Основные HTTP-методы

| Метод | Назначение |
|-------|-------------|
| `GET` | получить данные с сервера |
| `POST` | отправить новые данные |
| `PUT` | обновить существующие данные |
| `DELETE` | удалить данные |

В этом уроке сосредоточимся на `GET` — самом частом методе для парсинга.

## Первый запрос: получаем веб-страницу

In [ ]:
import requests

url = "https://www.example.com"
response = requests.get(url)

print(f"Статус код: {response.status_code}")  # 200 — успех
print(f"Первые 500 символов HTML:\n{response.text[:500]}")

In [ ]:
response.raise_for_status()

<Response [200]>

**Что здесь происходит?**
- `requests.get(url)` отправляет GET-запрос к серверу
- `response.status_code` — код ответа (200 = успех, 404 = не найдено, 500 = ошибка сервера)
- `response.text` — HTML-код страницы в виде текста

## Парсинг HTML с BeautifulSoup

Чтобы извлечь конкретные данные из HTML, нужно его распарсить. Лучшая связка: `requests` + `BeautifulSoup`.

Установка (если ещё не установлена):
```bash
pip install beautifulsoup4
```

## Практический пример: парсинг цитат

Возьмём сайт-песочницу [quotes.toscrape.com](http://quotes.toscrape.com), созданный специально для тренировки парсинга.

**Задача:** извлечь все цитаты и их авторов.

In [ ]:
texts_dict = dict()

texts_dict['pugs'] = get_texts('https://ajo-pet.ru/blog/mops/')

for elem in soup.find_all("a", class_="footer__nav-link meta-2"):
    url = 'https://ajo-pet.ru'+elem['href']
    texts_dict[url] = get_texts(url)

In [ ]:
texts_dict.keys()

dict_keys(['pugs', 'https://ajo-pet.ru/page/o-nas/', 'https://ajo-pet.ru/page/proizvodstvo/', 'https://ajo-pet.ru/page/calculator/', 'https://ajo-pet.ru/category/cats/', 'https://ajo-pet.ru/category/dogs/', 'https://ajo-pet.ru/ajo-vet/dogs/', 'https://ajo-pet.ru/ajo-vet/cats/', 'https://ajo-pet.ru/page/umnaya-eda/', 'https://ajo-pet.ru', 'https://ajo-pet.ru/news/', 'https://ajo-pet.ru/blog/'])

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_texts(url):
    response = requests.get(url)
    if response.status_code == 200:
        soup_ = BeautifulSoup(response.text, "html.parser")
        soup_texts_ = soup_.find_all("p", class_=None)
        return [x.text for x in soup_texts_ if len(x.text) > 0]

Как добавить header для более устойчивого скрепинга: https://sky.pro/wiki/python/otpravka-user-agent-cherez-python-requests-v-headers/

In [ ]:
import requests
from bs4 import BeautifulSoup

# 1. Отправляем запрос
url = "http://quotes.toscrape.com"
response = requests.get(url)

# 2. Проверяем, что всё прошло успешно
if response.status_code != 200:
    print("Не удалось загрузить страницу")
else:
    # 3. Передаём HTML в BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")

    # 4. Находим все блоки с цитатами (класс "quote")
    quotes = soup.find_all("div", class_="quote")

    # 5. Извлекаем данные из каждого блока
    for quote in quotes:
        text = quote.find("span", class_="text").text
        author = quote.find("small", class_="author").text
        print(f"«{text}» — {author}")
        print("-" * 50)

## Как BeautifulSoup ищет элементы?

| Метод | Что делает |
|-------|------------|
| `find_all("div", class_="quote")` | находит все `<div>` с классом `quote` |
| `.find("span", class_="text")` | внутри блока ищет `<span>` с классом `text` |
| `.find("small", class_="author")` | ищет `<small>` с классом `author` |
| `.text` | извлекает текст без HTML-тегов |

**Важно:** Если элемент не найден, `.find()` возвращает `None`, и вызов `.text` вызовет ошибку. Всегда добавляйте проверку:

In [ ]:
text_tag = quote.find("span", class_="text")
if text_tag:
    text = text_tag.text
else:
    text = "Текст не найден"

## Важное замечание: динамические сайты

`requests` получает только **исходный HTML** от сервера. Если сайт подгружает данные через JavaScript после загрузки страницы, то в `response.text` их не будет.

**Как распознать динамический сайт?**
- Данные видны в браузере, но отсутствуют в `response.text`.
- В инструментах разработчика (F12) данные приходят отдельными JSON-запросами.

**Что делать?**
- Искать API сайта (часто проще)
- Использовать `selenium` или `playwright` для управления браузером

**Простые сайты для тренировки с `requests`:**
- [quotes.toscrape.com](http://quotes.toscrape.com) — цитаты
- [books.toscrape.com](http://books.toscrape.com) — каталог книг

## Обработка ошибок

При парсинге реальных сайтов сеть ненадёжна. Добавим защиту от типичных проблем:

In [ ]:
import requests
from requests.exceptions import Timeout, ConnectionError, HTTPError
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com"

try:
    response = requests.get(url, timeout=10)  # ждём не более 10 секунд
    response.raise_for_status()               # проверяем на 4xx/5xx ошибки

    soup = BeautifulSoup(response.text, "html.parser")
    quotes = soup.find_all("div", class_="quote")

    for quote in quotes:
        text = quote.find("span", class_="text")
        author = quote.find("small", class_="author")

        quote_text = text.text if text else "Текст отсутствует"
        author_name = author.text if author else "Автор неизвестен"

        print(f"«{quote_text}» — {author_name}")
        print("-" * 50)

except Timeout:
    print("Ошибка: сервер не ответил за отведённое время.")
except ConnectionError:
    print("Ошибка: не удалось подключиться к сайту.")
except HTTPError as e:
    print(f"Ошибка HTTP: {e}")
except Exception as e:
    print(f"Произошла непредвиденная ошибка: {e}")

## Итог: простой план парсинга сайта

1. **Отправить GET-запрос** — `requests.get(url, timeout=...)`
2. **Проверить статус** — убедиться, что `status_code == 200`
3. **Передать HTML в BeautifulSoup** — `BeautifulSoup(response.text, "html.parser")`
4. **Найти нужные элементы** — `find()` или `find_all()` с селекторами
5. **Извлечь данные** — `.text` (текст) или `.get("href")` (ссылки)
6. **Обработать ошибки** — тайм-ауты, отсутствие элементов, сетевые проблемы

## Полный рабочий код (скопируйте и запустите)

```python
import requests
from bs4 import BeautifulSoup
from requests.exceptions import Timeout, ConnectionError, HTTPError

url = "http://quotes.toscrape.com"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "html.parser")
    quotes = soup.find_all("div", class_="quote")
    
    if not quotes:
        print("Цитаты не найдены. Возможно, сайт изменил структуру.")
    else:
        for quote in quotes:
            text = quote.find("span", class_="text")
            author = quote.find("small", class_="author")
            
            quote_text = text.text if text else "Текст отсутствует"
            author_name = author.text if author else "Автор неизвестен"
            
            print(f"«{quote_text}» — {author_name}")
            print("-" * 50)
            
except Timeout:
    print("Ошибка: сервер не ответил за отведённое время.")
except ConnectionError:
    print("Ошибка: не удалось подключиться к сайту.")
except HTTPError as e:
    print(f"Ошибка HTTP: {e}")
except Exception as e:
    print(f"Произошла непредвиденная ошибка: {e}")
```